# OpenPlaque — LCX Chamber Interface Midpoint Identity
Direct LA↔LV / RA↔RV chamber-surface midpoint interface. Single-kernel execution; no Python subprocess.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/LCX_Chamber_Interface_Midpoint_v1'
BRANCH = 'lcx-chamber-interface-midpoint-from-main'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_SOURCE_COMMIT = '6337c7e80bcbd69136068eb3dfa61b78573d90dd'
print('Branch:', BRANCH)
print('Pinned source commit:', EXPECTED_SOURCE_COMMIT)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo):
    shutil.rmtree(repo)
!git clone --depth 20 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout $EXPECTED_SOURCE_COMMIT
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)
assert HEAD == EXPECTED_SOURCE_COMMIT, (HEAD, EXPECTED_SOURCE_COMMIT)


In [ ]:
# Normal non-editable install, rebuilt from the pinned checkout.
%pip uninstall -y openplaque
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy


In [ ]:
# Purge any stale OpenPlaque modules left in this already-running kernel, then re-import.
import sys, importlib, inspect, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()

import openplaque
from openplaque import lcx_chamber_interface_midpoint_identity as exp

print('openplaque:', openplaque.__file__)
print('experiment module:', exp.__file__)
print('algorithm:', exp.ALGORITHM)

self_test_source = inspect.getsource(exp.synthetic_interface_self_test)
print(self_test_source)
assert 'np.linspace(0,20,201)' in self_test_source, 'Stale OpenPlaque module: patched 201-point self-test not loaded'
print('self-test:', exp.synthetic_interface_self_test())

rc = pytest.main(['-q','/content/OpenPlaque/tests/test_lcx_chamber_interface_midpoint_identity.py'])
if rc != 0:
    raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_left.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_left.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_right.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_right.nii.gz',
 root/'Joint_Three_Vessel_Template_Classifier_v1/LCX_joint_candidate_ranking.csv',
]+[root/f'Joint_Three_Vessel_Template_Classifier_v1/candidate_{i:02d}_source_path.csv' for i in range(1,6)]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing:
    raise FileNotFoundError('\n'.join(missing))


In [ ]:
# Direct same-kernel scientific execution. Full exceptions appear in this cell.
import gc
gc.collect()
result = exp.run(DRIVE_ROOT, OUTPUT_DIR)
print('STATUS:', result['summary']['status'])
print('CONTROLS:', result['summary']['chamber_interface_controls'])
print('DECISION:', result['summary']['decision'])
print('REPORT:', result['report'])
print('ZIP:', result['zip'])
